# 03. SBAS InSAR: Time-Series Inversion & Cumulative Subsidence Extraction

This notebook documents the complete Small Baseline Subset (SBAS) time-series InSAR inversion pipeline. The workflow ingests 245 unwrapped differential interferograms generated by ISCE2 topsStack (spanning 124 Sentinel-1 acquisition dates from January 2021 to April 2026) to reconstruct the multi-year spatiotemporal land subsidence field across the Sayung and coastal Demak alluvial plain.

### Key Processing Stages:
1. Environment Setup, Workspace Configuration, and Library Dependencies
2. Upstream ISCE2 Stack Integrity Audit (XML Metadata, Wavelength, UNW Phase Rasters)
3. Perpendicular Baseline Network Harmonization via Baseline Triangle Law
4. Hierarchical HDF5 Interferogram Stack Compilation (`ifgramStack.h5`)
5. Out-of-Core Row-Streaming Least-Squares SVD SBAS Inversion (`timeseries_manual.h5`)
6. Phase-to-Displacement Conversion and Time-Series Extraction at Sayung Core Hotspot
7. 2D Spatial Deformation Mapping and Cumulative Line-of-Sight (LOS) Visualization

## 1. Environment Initialization & Dependency Setup
Mounts the Google Drive storage repository and configures the numerical computing and geospatial raster drivers (NumPy, SciPy, GDAL, HDF5, and Matplotlib) required for out-of-core matrix inversion.

In [ ]:
import os
import sys
import glob
import re
import datetime
import h5py
import numpy as np
from scipy import linalg
from osgeo import gdal
import matplotlib.pyplot as plt
from google.colab import drive

drive.mount('/content/drive', force_remount=True)

BASE_DIR = '/content/drive/MyDrive/Thesis_Hands_On'
STACK_DIR = os.path.join(BASE_DIR, 'isce2_stack')
IFGRAM_DIR = os.path.join(STACK_DIR, 'merged/interferograms')
BASELINES_DIR = os.path.join(STACK_DIR, 'baselines')
MERGED_BSL_DIR = os.path.join(STACK_DIR, 'merged/baselines')
GEOM_DIR = os.path.join(STACK_DIR, 'merged/geom_reference')
OUTPUT_DIR = os.path.join(BASE_DIR, 'mintpy_output')
INPUTS_DIR = os.path.join(OUTPUT_DIR, 'inputs')

for directory in [MERGED_BSL_DIR, OUTPUT_DIR, INPUTS_DIR]:
    os.makedirs(directory, exist_ok=True)

print("Environment initialized successfully.")
print(f"Output Workspace: {OUTPUT_DIR}")

## 2. Upstream ISCE2 Stack Integrity Audit
Performs rigorous structural validation on upstream interferometric products before building the HDF5 tensor:
* Validates Sentinel-1 C-band radar wavelength ($\lambda \approx 0.05546576\text{ m}$) and orbit state parameters inside the primary reference XML header.
* Evaluates raster validity, null masks, and phase distribution on Band 2 unwrapped phase files (`filt_fine.unw`) to eliminate corrupted or empty grids.

In [ ]:
import xml.etree.ElementTree as ET

ref_xml_path = os.path.join(STACK_DIR, 'reference/IW2.xml')
assert os.path.exists(ref_xml_path), f"Reference XML not found at {ref_xml_path}"

tree = ET.parse(ref_xml_path)
root = tree.getroot()

wvl_node = root.find(".//property[@name='radar_wavelength']/value")
orb_node = root.find(".//property[@name='orbit_type']/value")

wavelength = float(wvl_node.text) if wvl_node is not None else 0.05546576
orbit_type = orb_node.text if orb_node is not None else "precise"

print(f"[Audit XML] Radar Wavelength: {wavelength:.8f} m | Orbit Type: {orbit_type}")
assert 0.055 <= wavelength <= 0.056, "Invalid Sentinel-1 C-band radar wavelength detected!"

sample_unw = glob.glob(os.path.join(IFGRAM_DIR, "*/filt_fine.unw"))[0]
ds_sample = gdal.Open(sample_unw)
band_phase = ds_sample.GetRasterBand(2)
arr_phase = band_phase.ReadAsArray()

valid_px = int(np.count_nonzero(~np.isnan(arr_phase)))
mean_phase = float(np.nanmean(arr_phase))
max_phase = float(np.nanmax(arr_phase))

print(f"[Audit UNW] Sample pair: {os.path.basename(os.path.dirname(sample_unw))}")
print(f"            Raster Dimensions: {ds_sample.RasterYSize} rows x {ds_sample.RasterXSize} columns")
print(f"            Valid Pixels: {valid_px} | Mean Phase: {mean_phase:.4f} rad | Max Phase: {max_phase:.4f} rad")
assert valid_px > 0 and max_phase != 0.0, "Empty or corrupted unwrapped phase raster detected!"

ds_sample = None

## 3. Perpendicular Baseline Harmonization via Baseline Triangle Law
Extracts spatial baseline offsets ($B_\perp$) relative to the master acquisition (`20210106`). The perpendicular baseline for any arbitrary interferometric pair $(t_1, t_2)$ is then computed using the closure loop:
$$B_\perp(t_1, t_2) = B_\perp(\text{master}, t_2) - B_\perp(\text{master}, t_1)$$
Formatted baseline records are subsequently written to disk for all 245 pairs.

In [ ]:
MASTER_DATE = '20210106'
master_bsl_dict = {MASTER_DATE: 0.0}

folders_bsl = [f for f in os.listdir(BASELINES_DIR) if os.path.isdir(os.path.join(BASELINES_DIR, f))]
for f in folders_bsl:
    parts = f.split('_')
    if len(parts) == 2:
        slave_date = parts[1]
        txt_path = os.path.join(BASELINES_DIR, f, f"{f}.txt")
        if os.path.exists(txt_path):
            with open(txt_path, 'r') as file:
                content = file.read()
            match = re.search(r"(?:P_BASELINE_TOP_HDR|Bperp \(average\):)\s*([\d\.-]+)", content)
            if match:
                master_bsl_dict[slave_date] = float(match.group(1))

print(f"Master baseline database compiled: {len(master_bsl_dict)} acquisition dates verified.")

ifgram_folders = sorted([os.path.basename(f) for f in glob.glob(os.path.join(IFGRAM_DIR, "*")) if os.path.isdir(f)])
total_pairs = len(ifgram_folders)
print(f"Total interferometric pairs detected: {total_pairs}")

pair_bsl_dict = {}
for pair in ifgram_folders:
    d1, d2 = pair.split('_')
    if d1 in master_bsl_dict and d2 in master_bsl_dict:
        bperp_calc = master_bsl_dict[d2] - master_bsl_dict[d1]
        pair_bsl_dict[pair] = bperp_calc

        out_txt = os.path.join(MERGED_BSL_DIR, f"{pair}.txt")
        with open(out_txt, 'w') as fout:
            fout.write(f"P_BASELINE_TOP_HDR {bperp_calc:.4f}\n")
            fout.write(f"P_BASELINE_BOTTOM_HDR {bperp_calc:.4f}\n")

print(f"Baseline Harmonization Complete: {len(pair_bsl_dict)} / {total_pairs} pairs synchronized.")

## 4. Hierarchical HDF5 Interferogram Stack Compilation (`ifgramStack.h5`)
Assembles the complete repository of unwrapped phase rasters, spatial coherence fields, connected component labels, temporal acquisition pairs, and perpendicular baselines into an indexed HDF5 container compressed with LZF.

In [ ]:
h5_stack_file = os.path.join(INPUTS_DIR, 'ifgramStack.h5')
if os.path.exists(h5_stack_file):
    os.remove(h5_stack_file)

unw_paths = sorted(glob.glob(os.path.join(IFGRAM_DIR, "*/filt_fine.unw")))
num_pairs = len(unw_paths)

ds_init = gdal.Open(unw_paths[0])
length, width = ds_init.RasterYSize, ds_init.RasterXSize
ds_init = None

print(f"Compiling ifgramStack.h5: {num_pairs} pairs | Spatial grid: {length} rows x {width} cols...")

with h5py.File(h5_stack_file, 'w') as f_h5:
    ds_unw = f_h5.create_dataset('unwrapPhase', (num_pairs, length, width), dtype='float32', compression='lzf')
    ds_cor = f_h5.create_dataset('coherence', (num_pairs, length, width), dtype='float32', compression='lzf')
    ds_conn = f_h5.create_dataset('connectComponent', (num_pairs, length, width), dtype='int16', compression='lzf')
    ds_bperp = f_h5.create_dataset('bperp', (num_pairs,), dtype='float32')
    ds_date = f_h5.create_dataset('date', (num_pairs, 2), dtype='S8')

    for i, path in enumerate(unw_paths):
        pair_name = os.path.basename(os.path.dirname(path))
        d1, d2 = pair_name.split('_')

        ds_ptr = gdal.Open(path)
        if ds_ptr:
            ds_unw[i] = ds_ptr.GetRasterBand(2).ReadAsArray()

            conn_path = path + ".conncomp"
            if os.path.exists(conn_path):
                ds_conn_ptr = gdal.Open(conn_path)
                ds_conn[i] = ds_conn_ptr.GetRasterBand(1).ReadAsArray()
                ds_conn_ptr = None
            else:
                ds_conn[i] = np.ones((length, width), dtype='int16')
            ds_ptr = None

        cor_path = path.replace('.unw', '.cor')
        if os.path.exists(cor_path):
            ds_cor_ptr = gdal.Open(cor_path)
            ds_cor[i] = ds_cor_ptr.GetRasterBand(1).ReadAsArray()
            ds_cor_ptr = None

        ds_date[i] = [d1.encode('ascii'), d2.encode('ascii')]
        ds_bperp[i] = pair_bsl_dict.get(pair_name, 0.0)

        if (i + 1) % 50 == 0 or (i + 1) == num_pairs:
            print(f"  Processed {i + 1}/{num_pairs} interferograms into HDF5 stack.")

    f_h5.attrs['PROCESSOR'] = 'isce'
    f_h5.attrs['WAVELENGTH'] = float(wavelength)
    f_h5.attrs['FILE_TYPE'] = 'interferograms'
    f_h5.attrs['UNIT'] = 'radian'

file_size_gb = os.path.getsize(h5_stack_file) / (1024**3)
print(f"Successfully generated ifgramStack.h5 ({file_size_gb:.2f} GB).")

## 5. Out-of-Core Row-Streaming SVD Least-Squares SBAS Inversion
Estimates the temporal phase series at each acquisition epoch via Singular Value Decomposition (SVD):
1. **Design Matrix Formulation:** Builds the temporal incidence matrix $A$ of shape $(M \times N)$ with $M=245$ interferometric pairs and $N=124$ distinct SAR acquisition dates.
2. **Rank-Deficiency Regularization:** Eliminates the primary reference column ($A_{\text{red}}$) and computes the Moore-Penrose pseudo-inverse $A_{\text{inv}} = (A_{\text{red}}^T A_{\text{red}})^{-1} A_{\text{red}}^T$.
3. **Row-Streaming Memory Optimization:** Inverts the 3D data volume row by row. This caps peak RAM utilization at $\sim 200\text{ MB}$ per iteration, circumventing Google Colab runtime crashes.

In [ ]:
output_ts_file = os.path.join(OUTPUT_DIR, 'timeseries_manual.h5')
if os.path.exists(output_ts_file):
    os.remove(output_ts_file)

with h5py.File(h5_stack_file, 'r') as f_in:
    dset_unw = f_in['unwrapPhase']
    dates = f_in['date'][:]
    num_pairs, length, width = dset_unw.shape

    unique_dates = np.unique(dates.flatten())
    num_dates = len(unique_dates)

    print(f"Inversion Setup: {num_pairs} pairs -> {num_dates} temporal acquisition epochs.")

    date_to_idx = {date: i for i, date in enumerate(unique_dates)}
    A = np.zeros((num_pairs, num_dates), dtype=np.float32)
    for i, (d1, d2) in enumerate(dates):
        A[i, date_to_idx[d1]] = -1.0
        A[i, date_to_idx[d2]] = 1.0

    A_red = A[:, 1:]
    print("Computing SVD Moore-Penrose Pseudo-Inverse of incidence matrix A...")
    A_inv = linalg.pinv(A_red).astype(np.float32)

    with h5py.File(output_ts_file, 'w') as f_out:
        ds_ts = f_out.create_dataset('timeseries', (num_dates, length, width),
                                     dtype='float32', compression='lzf')
        f_out.create_dataset('date', data=unique_dates)

        print(f"Executing out-of-core row-streaming inversion (Total rows: {length})...")

        for y in range(length):
            row_data = dset_unw[:, y, :]
            row_ts = np.dot(A_inv, row_data)
            ds_ts[1:, y, :] = row_ts
            if (y + 1) % 200 == 0 or (y + 1) == length:
                print(f"  Progress: Row {y + 1}/{length} inverted successfully.")

        ds_ts.attrs['UNIT'] = 'radian'
        ds_ts.attrs['WAVELENGTH'] = float(wavelength)

print(f"SBAS Inversion Complete! Result saved to: {output_ts_file}")

## 6. Phase-to-LOS Displacement Conversion & Time-Series Extraction
Converts temporal phase estimates ($\text{rad}$) into physical cumulative Line-of-Sight (LOS) displacement ($d_{\text{LOS}}$) in millimeters:
$$d_{\text{LOS}} = \Delta \phi \times \left( \frac{\lambda}{-4\pi} \right) \times 1000$$
Negative values denote downward deformation away from the satellite sensor (subsidence). The time series is extracted at the representative high-subsidence location in Sayung District (radar coordinates: $y=700, x=1300$).

In [ ]:
with h5py.File(output_ts_file, 'r') as f_ts:
    ts_data = f_ts['timeseries']
    dates_str = [d.decode('utf-8') if isinstance(d, bytes) else str(d) for d in f_ts['date'][:]]

    target_y, target_x = 700, 1300
    pixel_phase = ts_data[:, target_y, target_x]

    conversion_factor = (float(wavelength) * 1000.0) / (-4.0 * np.pi)
    displacement_mm = pixel_phase * conversion_factor

plt.figure(figsize=(12, 5), dpi=150)
plt.plot(dates_str, displacement_mm, marker='o', markersize=3.5, color='#d62728', linewidth=1.5, label='Sayung Core (Pixel [700, 1300])')
plt.axhline(0, color='black', linestyle='--', linewidth=0.8, alpha=0.7)
plt.title(f'SBAS-InSAR Cumulative LOS Displacement (Sayung Hotspot: Pixel [{target_y}, {target_x}])', fontsize=12, pad=12)
plt.xlabel('SAR Acquisition Date (YYYYMMDD)', fontsize=10)
plt.ylabel('Cumulative LOS Displacement (mm)', fontsize=10)
plt.xticks(dates_str[::12], rotation=45, ha='right', fontsize=8)
plt.grid(True, linestyle=':', alpha=0.6)
plt.legend(loc='lower left', frameon=True)
plt.tight_layout()
plt.show()

total_subsidence = displacement_mm[-1] - displacement_mm[0]
print(f"Total Cumulative Displacement at ({target_y}, {target_x}): {total_subsidence:.2f} mm")

## 7. 2D Spatial Cumulative Subsidence Mapping (2021–2026)
Visualizes the regional 2D spatial distribution of total cumulative subsidence across the Kendal–Semarang–Demak coastal belt as of April 2026 relative to the January 2021 baseline.

In [ ]:
with h5py.File(output_ts_file, 'r') as f_ts:
    final_phase_map = f_ts['timeseries'][-1, :, :]
    final_disp_map = final_phase_map * conversion_factor

plt.figure(figsize=(10, 8), dpi=150)
im = plt.imshow(final_disp_map, cmap='jet_r', vmin=-400, vmax=50)

plt.plot(target_x, target_y, marker='^', markersize=10, color='black', markeredgecolor='white', label=f'Sayung Core ({target_y}, {target_x})')

cbar = plt.colorbar(im, fraction=0.046, pad=0.04)
cbar.set_label('Cumulative LOS Displacement (mm)', fontsize=10)
plt.title('Cumulative Land Subsidence Map: Sayung–Demak Corridor (2021–2026)', fontsize=12, pad=12)
plt.xlabel('Range Grid Index (Pixels)', fontsize=10)
plt.ylabel('Azimuth Grid Index (Pixels)', fontsize=10)
plt.legend(loc='upper right')
plt.tight_layout()
plt.show()